# Module 11: garak, PyRIT and promptfoo against ToyScreen
**Your own, intentionally vulnerable lab target only. FICTIONAL DATA.** Everything runs on `127.0.0.1` inside this runtime.

In [ ]:
import os, subprocess
if not os.path.exists("scripts/start_lab_assistant.sh"):
    if not os.path.exists("ai-hiring-audit-lab"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/anthonyhammock/ai-hiring-audit-lab"], check=True)
    os.chdir("ai-hiring-audit-lab/m11-redteam-tools")
%pip install -q -r ../toyscreen/requirements.txt
!bash scripts/start_lab_assistant.sh

## Lesson 02: garak (large install: a few minutes)

In [ ]:
%pip install -q garak==0.17.1
!python -m garak -t test.Repeat -S probes.encoding.InjectBase64 -g 1
!python -m garak --config garak/toyscreen_rest.yaml --report_prefix toyscreen_baseline

## Lesson 03: PyRIT

In [ ]:
%pip install -q pyrit==1.1.0
os.environ.update(OPENAI_CHAT_ENDPOINT="http://127.0.0.1:8080/v1", OPENAI_CHAT_KEY="not-needed", OPENAI_CHAT_MODEL="toyscreen")
!python pyrit/attack_toyscreen.py --repeats 10

## Lesson 04: promptfoo regression suite

In [ ]:
!PROMPTFOO_PASS_RATE_THRESHOLD=100 npx -y promptfoo@0.125.1 eval -c promptfoo/redteam/regression.yaml --repeat 10 --no-cache; echo "exit code $?"
!bash scripts/stop_lab_assistant.sh
!TOYSCREEN_MODE=hardened bash scripts/start_lab_assistant.sh
!PROMPTFOO_PASS_RATE_THRESHOLD=100 npx -y promptfoo@0.125.1 eval -c promptfoo/redteam/regression.yaml --repeat 10 --no-cache; echo "exit code $?"